## 3. Baseline Models

In [ ]:
from pytabkit import RealMLP_TD_Classifier
from pytabkit import TabM_D_Classifier

from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
import pandas as pd

import os
import sys
sys.path.append(os.path.abspath('..'))

from src.models import train_pipline
from src.utils import show_confusion_matrix

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
raw_path = os.path.dirname(os.path.abspath('.'))
df_FE_train_path = os.path.join(raw_path, "df_FE_train.csv")
df_FE_test_path = os.path.join(raw_path, "df_FE_test.csv")

df_FE = pd.read_csv(df_FE_train_path)

df_FE.shape

In [ ]:
X = df_FE.drop(['Stress_Level'], axis=1)
y = df_FE['Stress_Level']

In [ ]:
X

In [ ]:
X.columns

In [ ]:
y

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

target_names=['0','1']

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
y_resampled.value_counts()

In [ ]:
ET_model,y_preds,y_probas = train_pipline(X_resampled,y_resampled,ExtraTreesClassifier,cv=cv)
show_confusion_matrix(y_resampled,y_preds,target_names)
print(classification_report(y_resampled,y_preds,target_names=target_names))

In [ ]:
LGBM_model,y_preds,y_probas = train_pipline(X_resampled,y_resampled,LGBMClassifier,cv=cv)
show_confusion_matrix(y_resampled,y_preds,target_names)
print(classification_report(y_resampled,y_preds,target_names=target_names))

In [ ]:
XGB_model,y_preds, y_probas = train_pipline(X_resampled,y_resampled,XGBClassifier,cv=cv)
show_confusion_matrix(y_resampled,y_preds,target_names)
print(classification_report(y_resampled,y_preds,target_names=target_names))

In [ ]:
fi = pd.DataFrame({'features': X.columns,
                   'importances': ET_model.feature_importances_})

fi.sort_values('importances',ascending=False)

In [ ]:
fi = pd.DataFrame({'features': X.columns,
                   'importances': LGBM_model.feature_importances_})

fi.sort_values('importances',ascending=False)

In [ ]:
fi = pd.DataFrame({'features': X.columns,
                   'importances': XGB_model.feature_importances_})

fi.sort_values('importances',ascending=False)